<a href="https://colab.research.google.com/github/romisap/invoice-e4b-finetuning-worksheet/blob/main/worksheet/E4B_Invoice_Finetuning_Worksheet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# E4B Invoice to JSON — Fine-Tuning Worksheet

From messy invoice text to strict JSON — by teaching a small open model a new trick, in about 30-40 minutes on a free T4 GPU.

**What you will build:** the core of a real SAP Accounts-Payable automation bot. Vendors send invoices as formal emails, one-line notes, tabular blocks, even WhatsApp Hinglish. You will teach Gemma E4B to always answer with the same strict 8-field JSON.

**What you will learn:** prompting vs fine-tuning, LoRA/QLoRA, building a dataset with code, and how to evaluate properly (spoiler: loss is not the metric).

**Real reference run** (Kaggle T4 — see repo results/ and eval/): base model got **0/20** perfect extractions. After fine-tuning on just 80 examples for ~3 minutes: **17/20** perfect, average field accuracy **10.6% to 97.5%**.

**How to run:** Open in Google Colab, Runtime, Change runtime type, T4 GPU (or run on Kaggle). Everything is generated by code — no downloads, no API keys.

## Theory 1 — Prompting vs Fine-Tuning

Two ways to get an LLM to do a task:

| | Prompting (zero/few-shot) | Fine-tuning |
|---|---|---|
| How | Instructions + examples inside the prompt | Show input-to-output pairs, update a small part of the weights |
| Weights | Untouched | Slightly changed (adapters) |
| Best for | Open-ended, general tasks, quick experiments | Narrow, repetitive, strict-format tasks |
| Cost per call | Long prompt every time | Short prompt — behaviour is now the default |
| Failure mode | Forgets the format, invents fields | Only as good as your dataset |

**Mental model:** fine-tuning does not add new knowledge. It locks a behaviour and a format into the weights. That is exactly why this task is perfect for it: 8 fixed fields, strict JSON, small domain.

**When NOT to fine-tune:** if the format changes every week, if the model must reason or cite sources, or if few-shot prompting already gives you 100%.

## Theory 2 — LoRA and QLoRA (how a free T4 can train a 4B model)

Full fine-tuning updates every weight — billions of numbers, data-centre GPU. Two tricks make it fit in 16 GB:

1. **LoRA (Low-Rank Adaptation):** freeze the whole model, attach tiny trainable adapter matrices (rank r = 16) at 7 attention/MLP points (q, k, v, o, gate, up, down projections). Only the adapters learn — a small fraction of all parameters. The base stays untouched, so you can swap adapters per task.
2. **QLoRA:** keep the frozen base in 4-bit and train the adapters in 32-bit. A much smaller memory footprint — that is what lets a free T4 handle a 4B model.

This worksheet uses: r = 16, alpha = 16, learning rate 2e-4, 3 epochs, batch 2 with gradient accumulation 8. The unsloth library compiles fast kernels, so the whole training is 15 optimizer steps, about 3 minutes on a T4.

In [3]:
import subprocess
r = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True)
info = r.stdout.decode().strip()
if info:
    print("GPU detected:", info)
else:
    print("No GPU detected! In Colab: Runtime -> Change runtime type -> T4 GPU")

GPU detected: Tesla T4, 15360 MiB


In [4]:
import os
os.environ["UNSLOTH_FORCE_FLOAT32"] = "1"  # Gemma E4B quirk: float16 breaks training on T4
os.system("pip install --pre --upgrade --no-deps unsloth")
os.system("pip install --no-deps --force-reinstall git+https://github.com/unslothai/unsloth-zoo.git")
os.system("pip install --no-deps trl peft accelerate bitsandbytes datasets")
print("installs done — 2-4 minutes")

installs done — 2-4 minutes


In [5]:
import torch
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/gemma-4-E4B-it",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
    device_map={"": 0},
)

probe = tokenizer.apply_chat_template([{"role": "user", "content": "hi"}], tokenize=False)
print("TEMPLATE PROBE:", repr(probe))

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.10.6: Fast Gemma4 patching. Transformers: 5.18.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Using float16 precision for gemma4 won't work! Using float32.


Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

TEMPLATE PROBE: '<bos><|turn>user\nhi<turn|>\n'


## Theory 3 — The Task: Invoice to JSON (SAP AP framing)

In a real Accounts-Payable flow (SAP FI), every invoice must become structured data for the 3-way match: purchase order vs goods receipt vs invoice. A human typing 8 fields per invoice is exactly the job automation eats first.

**The schema (always all 8 keys):** vendor, invoice_number, invoice_date, base_amount, gst_amount, total_amount, po_reference, payment_terms_days

**4 input formats** (25 cases each in the dataset): formal email, one-line short note, tabular block, WhatsApp Hinglish.

**3 edge cases baked in on purpose:**
- **no_po** — no PO reference on the invoice. Correct answer: null. A model that invents a PO number would create a false match in SAP. We track it as a separate safety metric: invented_po must stay 0.
- **incl_total_only** — only the GST-inclusive total is given. Correct answer requires computing base = total / (1 + rate/100). This is reasoning, not copying.
- **missing_terms** — payment terms missing: null.

**Golden rule: extract, never invent.** Missing means null, not a guess.

In [6]:
# The dataset is a script, not a file: seed 42, fully reproducible.
# This is the exact generator from the reference run (data/make_invoice_dataset.py).
import json, random, os, hashlib

SEED = 42
OUT = "./data"
os.makedirs(OUT, exist_ok=True)
random.seed(SEED)
NL = chr(10)

VENDORS = ["Sharma Traders Pvt Ltd", "Gupta Enterprises", "Verma Supplies Co",
    "Mehta and Sons", "Kiran Wholesale Mart", "Deepak Materials Pvt Ltd",
    "Bansal Packaging Industries", "Nanda Electricals", "Prakash Hardware Store",
    "Sundaram Textiles Ltd", "Kapoor Imports", "Reddy Agro Supplies",
    "Chawla Chemicals Pvt Ltd", "Iyer Logistics Solutions", "Grover Steel Traders"]
GST_RATES = [5, 12, 18, 28]
TERMS_OPTIONS = [15, 30, 45, 60]
STYLES = ["email", "short", "tabular", "whatsapp"]
MONTHS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

def rupee(x):
    return "Rs." + format(int(x), ",")

def rand_date():
    return "2026-%02d-%02d" % (random.randint(1, 12), random.randint(1, 28))

def human_date(iso):
    y, m, d = iso.split("-")
    return "%s %s %s" % (d, MONTHS[int(m) - 1], y)

def render(style, vendor, inv_no, iso, base, gst, total, rate, po, terms, incl_only):
    total_s = rupee(total)
    base_s = rupee(base)
    gst_s = rupee(gst)
    if style == "email":
        s = "Dear Sir, please find attached invoice " + inv_no + " from " + vendor + " dated " + human_date(iso) + ". "
        if incl_only:
            s += "Total payable " + total_s + " inclusive of GST at " + str(rate) + "%. "
        else:
            s += "Base amount " + base_s + " plus GST " + gst_s + ", total payable " + total_s + ". "
        if po:
            s += "PO ref " + po + ". "
        if terms:
            s += "Payment terms " + str(terms) + " days."
        return s
    if style == "short":
        s = "Invoice " + inv_no + " | " + vendor + " | " + human_date(iso) + " | "
        if incl_only:
            s += "Total " + total_s + " (incl. GST " + str(rate) + "%)"
        else:
            s += "Base " + base_s + " + GST " + str(rate) + "% = " + total_s
        if po:
            s += " | PO " + po
        if terms:
            s += " | Terms " + str(terms) + "d"
        return s
    if style == "tabular":
        lines = ["Vendor: " + vendor, "Invoice No: " + inv_no, "Date: " + human_date(iso)]
        if incl_only:
            lines.append("Total (incl. GST " + str(rate) + "%): " + total_s)
        else:
            lines.append("Base Amount: " + base_s)
            lines.append("GST (" + str(rate) + "%): " + gst_s)
            lines.append("Total: " + total_s)
        if po:
            lines.append("PO: " + po)
        if terms:
            lines.append("Terms: " + str(terms) + " days")
        return NL.join(lines)
    s = "bhaiya invoice aa gaya hai - " + inv_no + ", " + vendor + ", date " + human_date(iso) + ". "
    if incl_only:
        s += "Total " + total_s + " (GST " + str(rate) + "% included). "
    else:
        s += "Total " + total_s + " hai, usme GST " + gst_s + " shamil hai. "
    if po:
        s += "PO number " + po + " hai. "
    if terms:
        s += "Payment " + str(terms) + " din ke andar."
    return s

def make_case(i, used_inv):
    vendor = VENDORS[i % len(VENDORS)]
    while True:
        inv_no = "INV-2026-" + str(random.randint(10000, 99999))
        if inv_no not in used_inv:
            used_inv.add(inv_no)
            break
    iso = rand_date()
    rate = GST_RATES[i % len(GST_RATES)]
    base = random.randrange(50, 2000) * 100
    gst = base * rate // 100
    total = base + gst
    incl_only = (i % 7 == 3)
    no_po = (i % 5 == 4)
    po = None
    if not no_po:
        po = "55" + str(random.randint(1000000, 9999999))
    missing_terms = (i % 20 == 9)
    terms = None
    if not missing_terms:
        terms = random.choice(TERMS_OPTIONS)
    style = STYLES[i % 4]
    text = render(style, vendor, inv_no, iso, base, gst, total, rate, po, terms, incl_only)
    answer = json.dumps({
        "vendor": vendor,
        "invoice_number": inv_no,
        "invoice_date": iso,
        "base_amount": float(base),
        "gst_amount": float(gst),
        "total_amount": float(total),
        "po_reference": po,
        "payment_terms_days": terms,
    }, indent=2)
    edges = []
    if no_po:
        edges.append("no_po")
    if incl_only:
        edges.append("incl_total_only")
    if missing_terms:
        edges.append("missing_terms")
    if not edges:
        edges.append("none")
    return {"messages": [
        {"role": "user", "content": "Extract the invoice fields as JSON:" + NL + NL + text},
        {"role": "assistant", "content": answer},
    ], "meta": {"case": i, "format": style, "edges": ",".join(edges)}}

used_inv = set()
golden = [make_case(i, used_inv) for i in range(100)]

SCHEMA = set(["vendor", "invoice_number", "invoice_date", "base_amount", "gst_amount", "total_amount", "po_reference", "payment_terms_days"])
errors = []
prompts = set()
for i, p in enumerate(golden):
    try:
        a = json.loads(p["messages"][1]["content"])
    except Exception:
        errors.append("case " + str(i) + ": assistant JSON unparseable")
        continue
    if set(a.keys()) != SCHEMA:
        errors.append("case " + str(i) + ": schema mismatch")
    if abs(a["base_amount"] + a["gst_amount"] - a["total_amount"]) > 0.001:
        errors.append("case " + str(i) + ": base+gst != total")
    h = hashlib.md5(p["messages"][0]["content"].encode()).hexdigest()
    if h in prompts:
        errors.append("case " + str(i) + ": duplicate prompt")
    prompts.add(h)
if errors:
    print("VALIDATION FAILED:")
    for e in errors:
        print(" - " + e)
    raise SystemExit(1)

srng = random.Random(42)
idx = list(range(100))
srng.shuffle(idx)
train = [golden[i] for i in idx[:80]]
test = [golden[i] for i in idx[80:]]

def write_jsonl(name, rows):
    with open(os.path.join(OUT, name), "w") as f:
        for r in rows:
            f.write(json.dumps(r) + NL)

write_jsonl("invoice_golden_v1_100.jsonl", golden)
write_jsonl("invoice_train_v1_80.jsonl", train)
write_jsonl("invoice_test_v1_20.jsonl", test)

fmt = {}
edges = {}
for p in golden:
    fmt[p["meta"]["format"]] = fmt.get(p["meta"]["format"], 0) + 1
    for e in p["meta"]["edges"].split(","):
        edges[e] = edges.get(e, 0) + 1

readme = """# Invoice Extract Dataset v1 (Golden, 100 handwritten-quality cases)

Task: messy invoice text (email / short / tabular / WhatsApp style) -> strict JSON with
vendor, invoice_number, invoice_date, base_amount, gst_amount, total_amount,
po_reference, payment_terms_days.

Files:
- invoice_golden_v1_100.jsonl : master golden dataset (100 cases, with meta)
- invoice_train_v1_80.jsonl  : train split (80 cases)
- invoice_test_v1_20.jsonl   : held-out test split (20 cases) - never train on these
- make_invoice_dataset.py    : generator (seed 42, fully reproducible)

Edge cases by design: no_po (PO missing -> null), incl_total_only (base must be
computed from inclusive total), missing_terms (terms missing -> null).

Validation: 100/100 assistant outputs parse as JSON, schema exact, base+gst=total
holds, 0 duplicate prompts.

Created: 2026-10-16 for the E4B invoice-extraction teaching track.
Regenerate: python3 make_invoice_dataset.py
"""

with open(os.path.join(OUT, "README.md"), "w") as f:
    f.write(readme)

print("GOLDEN DATASET OK - 100 cases, validation passed")
print("Formats: " + json.dumps(fmt))
print("Edges:   " + json.dumps(edges))
print("Train: " + str(len(train)) + " | Test: " + str(len(test)))
print("---- SAMPLE (case 0) ----")
print("USER:")
print(golden[0]["messages"][0]["content"])
print("ASSISTANT:")
print(golden[0]["messages"][1]["content"])


GOLDEN DATASET OK - 100 cases, validation passed
Formats: {"email": 25, "short": 25, "tabular": 25, "whatsapp": 25}
Edges:   {"none": 69, "incl_total_only": 14, "no_po": 20, "missing_terms": 5}
Train: 80 | Test: 20
---- SAMPLE (case 0) ----
USER:
Extract the invoice fields as JSON:

Dear Sir, please find attached invoice INV-2026-93810 from Sharma Traders Pvt Ltd dated 01 Feb 2026. Base amount Rs.156,800 plus GST Rs.7,840, total payable Rs.164,640. PO ref 555614226. Payment terms 30 days.
ASSISTANT:
{
  "vendor": "Sharma Traders Pvt Ltd",
  "invoice_number": "INV-2026-93810",
  "invoice_date": "2026-02-01",
  "base_amount": 156800.0,
  "gst_amount": 7840.0,
  "total_amount": 164640.0,
  "po_reference": "555614226",
  "payment_terms_days": 30
}


## Theory 4 — How to Evaluate (and why NOT loss)

Training loss measures how surprised the model was by the next token — not whether the output is business-correct. A model can have low loss and still put a decimal point in the wrong place.

Our metrics (same harness before and after, temperature 0 so runs are deterministic):
- **json_valid** — output parses as JSON at all
- **schema_ok** — all 8 keys present
- **all_fields_exact** — every field matches gold (numbers with 0.01 tolerance)
- **per-field accuracy** — which fields are weak?
- **invented_po** — PO invented when gold is null (must stay 0; the safety metric)

Run the harness below on the base model BEFORE training. Expect it to be messy — that is the point.

In [7]:
import json, os

SCHEMA = ["vendor", "invoice_number", "invoice_date", "base_amount", "gst_amount",
          "total_amount", "po_reference", "payment_terms_days"]

def load_jsonl(path):
    rows = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def extract_json(text):
    s = text.find("{")
    if s == -1:
        return None
    depth = 0
    for i in range(s, len(text)):
        ch = text[i]
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                try:
                    return json.loads(text[s:i+1])
                except Exception:
                    return None
    return None

test_pairs = load_jsonl("data/invoice_test_v1_20.jsonl")
print("test cases:", len(test_pairs))

def run_eval(model_obj, tag):
    json_valid = 0
    schema_ok = 0
    all_ok = 0
    invented_po = 0
    field_ok = {k: 0 for k in SCHEMA}
    cases = []
    inner = getattr(tokenizer, "tokenizer", tokenizer)
    for n, c in enumerate(test_pairs):
        user = c["messages"][0]["content"]
        gold = json.loads(c["messages"][1]["content"])
        prompt = tokenizer.apply_chat_template(
            [{"role": "user", "content": user}], tokenize=False, add_generation_prompt=True)
        ids = inner(prompt, return_tensors="pt").input_ids.to("cuda")
        inputs = {"input_ids": ids, "attention_mask": torch.ones_like(ids)}
        try:
            out = model_obj.generate(**inputs, max_new_tokens=512, do_sample=False)
            reply = inner.decode(out[0][ids.shape[1]:], skip_special_tokens=True)
        except Exception as e:
            reply = "ERROR: " + str(e)
        pred = extract_json(reply)
        row = {"case": n, "json_valid": pred is not None, "reply_head": reply[:200]}
        if pred is not None:
            json_valid += 1
            ok = 0
            if set(pred.keys()) == set(SCHEMA):
                schema_ok += 1
            for k in SCHEMA:
                pv = pred.get(k)
                gv = gold.get(k)
                if isinstance(gv, (int, float)) and isinstance(pv, (int, float)) and not isinstance(gv, bool):
                    same = abs(float(pv) - float(gv)) < 0.01
                else:
                    same = (pv == gv)
                if same:
                    field_ok[k] += 1
                    ok += 1
            if ok == len(SCHEMA):
                all_ok += 1
            if gold.get("po_reference") is None and pred.get("po_reference") is not None:
                invented_po += 1
            row["ok_fields"] = ok
            row["pred"] = pred
        row["gold"] = gold
        cases.append(row)
        print("case " + str(n) + " (" + tag + "): json_valid=" + str(row["json_valid"]) +
              " ok_fields=" + str(row.get("ok_fields", 0)) + "/" + str(len(SCHEMA)))
    total = len(test_pairs)
    summary = {
        "tag": tag, "total": total,
        "json_valid": json_valid, "schema_ok": schema_ok,
        "all_fields_exact": all_ok,
        "invented_po_when_missing": invented_po,
        "field_accuracy": {k: round(field_ok[k] / total, 3) for k in SCHEMA},
    }
    os.makedirs("results", exist_ok=True)
    with open("results/eval_" + tag + "_results.json", "w") as f:
        json.dump({"summary": summary, "cases": cases}, f, indent=2)
    print(json.dumps(summary, indent=2))
    return summary

print("Running BEFORE (zero-shot baseline) — expect 5-10 minutes on a T4")
results_before = run_eval(model, "before")

test cases: 20
Running BEFORE (zero-shot baseline) — expect 5-10 minutes on a T4
case 0 (before): json_valid=True ok_fields=4/8
case 1 (before): json_valid=True ok_fields=3/8
case 2 (before): json_valid=True ok_fields=1/8
case 3 (before): json_valid=True ok_fields=4/8
case 4 (before): json_valid=True ok_fields=1/8
case 5 (before): json_valid=True ok_fields=4/8
case 6 (before): json_valid=True ok_fields=2/8
case 7 (before): json_valid=True ok_fields=3/8
case 8 (before): json_valid=True ok_fields=3/8
case 9 (before): json_valid=True ok_fields=5/8
case 10 (before): json_valid=True ok_fields=2/8
case 11 (before): json_valid=True ok_fields=1/8
case 12 (before): json_valid=True ok_fields=1/8
case 13 (before): json_valid=True ok_fields=4/8
case 14 (before): json_valid=True ok_fields=1/8
case 15 (before): json_valid=True ok_fields=1/8
case 16 (before): json_valid=True ok_fields=3/8
case 17 (before): json_valid=True ok_fields=1/8
case 18 (before): json_valid=True ok_fields=3/8
case 19 (before):

## Theory 5 — The Training Config, Line by Line

- **train_on_responses_only** with markers around the user and model turns: loss is computed only on the assistant answer, not on the question. The model learns to answer, not to parrot invoices.
- **r = 16, alpha = 16:** adapter rank and scaling — small adapters, enough capacity for a strict format.
- **learning rate 2e-4:** standard for LoRA — much higher than full fine-tuning, because only the adapters train.
- **3 epochs, batch 2 x grad-accum 8:** 80 examples make just 15 optimizer steps. Small data needs few steps; more epochs would start memorising instead of generalising.
- **UNSLOTH_FORCE_FLOAT32:** a Gemma E4B quirk — float16 breaks training; unsloth falls back to float32 adapters.

Reference run: 15 steps in 175.8 seconds on a T4, loss 0.475 to 0.053 to 0.022 across the 3 epochs.

In [ ]:
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

model = FastLanguageModel.get_peft_model(
    model,
    r=16, alpha=16, dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
)

train_pairs = load_jsonl("data/invoice_train_v1_80.jsonl")
print("train pairs:", len(train_pairs))
train_ds = Dataset.from_list([{"messages": p["messages"]} for p in train_pairs])

trainer = train_on_responses_only(
    SFTTrainer(
        model=model,
        train_dataset=train_ds,
        args=SFTConfig(
            per_device_train_batch_size=2,
            gradient_accumulation_steps=8,
            num_train_epochs=3,
            learning_rate=2e-4,
            warmup_ratio=0.03,
            logging_steps=5,
            output_dir="./out",
            report_to="none",
        ),
        tokenizer=tokenizer,
    ),
    instruction_part="<|turn>user",
    response_part="<|turn>model",
)

trainer.train()
print("TRAIN HISTORY LAST:", trainer.state.log_history[-1])

model.save_pretrained("./invoice_e4b_lora")
tokenizer.save_pretrained("./invoice_e4b_lora")
print("LoRA adapter saved to ./invoice_e4b_lora — about 3 minutes total on a T4")

In [9]:
FastLanguageModel.for_inference(model)
print("Running AFTER (fine-tuned) — same 20 test cases, same harness")
results_after = run_eval(model, "after")

Running AFTER (fine-tuned) — same 20 test cases, same harness
case 0 (after): json_valid=True ok_fields=8/8
case 1 (after): json_valid=True ok_fields=8/8
case 2 (after): json_valid=True ok_fields=8/8
case 3 (after): json_valid=True ok_fields=8/8
case 4 (after): json_valid=True ok_fields=8/8
case 5 (after): json_valid=True ok_fields=8/8
case 6 (after): json_valid=True ok_fields=8/8
case 7 (after): json_valid=True ok_fields=8/8
case 8 (after): json_valid=True ok_fields=8/8
case 9 (after): json_valid=True ok_fields=8/8
case 10 (after): json_valid=True ok_fields=8/8
case 11 (after): json_valid=True ok_fields=8/8
case 12 (after): json_valid=True ok_fields=6/8
case 13 (after): json_valid=True ok_fields=8/8
case 14 (after): json_valid=True ok_fields=7/8
case 15 (after): json_valid=True ok_fields=8/8
case 16 (after): json_valid=True ok_fields=8/8
case 17 (after): json_valid=True ok_fields=7/8
case 18 (after): json_valid=True ok_fields=8/8
case 19 (after): json_valid=True ok_fields=8/8
{
  "tag

In [ ]:
def pctf(x):
    return str(round(100.0 * x, 1)) + "%"

total = results_before["total"]

def cnt(x):
    return str(x) + "/" + str(total) + " (" + pctf(x / total) + ")"

print("| Metric | Before | After |")
print("|---|---|---|")
print("| Valid JSON | " + cnt(results_before["json_valid"]) + " | " + cnt(results_after["json_valid"]) + " |")
print("| Full schema (all 8 keys) | " + cnt(results_before["schema_ok"]) + " | " + cnt(results_after["schema_ok"]) + " |")
print("| All 8 fields exact | " + cnt(results_before["all_fields_exact"]) + " | " + cnt(results_after["all_fields_exact"]) + " |")
print("| Invented PO (safety, must be 0) | " + str(results_before["invented_po_when_missing"]) + " | " + str(results_after["invented_po_when_missing"]) + " |")
print()
print("| Field | Before | After |")
print("|---|---|---|")
for k in SCHEMA:
    print("| " + k + " | " + pctf(results_before["field_accuracy"][k]) + " | " + pctf(results_after["field_accuracy"][k]) + " |")
print()
avg_b = sum(results_before["field_accuracy"].values()) / len(SCHEMA)
avg_a = sum(results_after["field_accuracy"].values()) / len(SCHEMA)
print("Average field accuracy:", pctf(avg_b), "->", pctf(avg_a))

## Reference Results (real run, Kaggle T4)

| Metric | Before (zero-shot) | After (80 pairs, 3 min training) |
|---|---|---|
| Valid JSON | 55% (11/20) | 100% (20/20) |
| Full schema | 0% (0/20) | 65% (13/20) |
| All 8 fields exact | 0% (0/20) | 85% (17/20) |
| Invented PO | 0 | 0 |
| Avg field accuracy | 10.6% | 97.5% |

**Timing:** setup + model load about 3 min, training 15 steps in 175.8 s, in-kernel eval about 10 min, total kernel about 16 min.

**Loss curve:** epoch 1: 0.475, epoch 2: 0.053, epoch 3: 0.022. The 10x drop in epoch 2 is the model getting the format.

**Honest notes (good classroom discussion):**
- Full schema is only 65% after fine-tuning: the model sometimes omits keys whose correct value is null (e.g. po_reference: null). The values are right, a key is missing. More no_po/missing_terms emphasis in the training data would likely fix it — see exercises.
- GST (85%) and base amount (95%) are the weak fields: exactly the incl_total_only cases that need computing, not copying.
- Your live numbers may differ slightly (different GPU / library versions). The shape of the before/after gap is the lesson.

## Exercises

1. **Fix the null omission.** Raise the share of no_po and missing_terms cases in the generator, regenerate, retrain, and check whether schema_ok climbs above 65%.
2. **GST accuracy.** incl_total_only cases are the weak spot (GST 85%, base 95%). Increase their frequency and see if field accuracy reaches 100%.
3. **Data efficiency.** Train on only the first 20 examples instead of 80. How much accuracy do you lose? Where are the diminishing returns?
4. **Add a 5th format.** Write an OCR-scanned-invoice style (broken lines, stray spaces). Regenerate, retrain, compare per-format accuracy.
5. **Push further.** Try a bigger model on a bigger GPU. On a T4 the 12B variant does not fit — a good lesson: the recipe is the value, the model size is just a knob.

## Repo Map and Further Reading

- **worksheet/** — this notebook
- **data/** — the generator (seed 42, reproducible) and the exact golden/train/test JSONL files used in the reference run
- **eval/** — the two real result files (before/after) and the standalone llama.cpp baseline script used on the server
- **training/** — the exact Kaggle kernel script of the reference run
- **results/** — RESULTS.md with full tables, timing and methodology

Further reading: the LoRA paper (Hu et al., 2021), the QLoRA paper (Dettmers et al., 2023), unsloth documentation, HuggingFace PEFT docs.

**What next?** You now have the full loop: dataset, baseline, train, measure, compare. The same recipe scales from invoices to any messy-text-to-JSON task.